In [1]:
# FINAL HOUSE PRICE PREDICTION NOTEBOOK

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_log_error

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

train_df = pd.read_csv("Ames_train.csv")
train_df.rename(columns=lambda x: x.strip().replace(' ', ''), inplace=True)
test_df.rename(columns=lambda x: x.strip().replace(' ', ''), inplace=True)
test_df = pd.read_csv("Ames_test.csv")
train_df.rename(columns=lambda x: x.strip().replace(' ', ''), inplace=True)
test_df.rename(columns=lambda x: x.strip().replace(' ', ''), inplace=True)

def add_features(df, neighborhood_median=None):
    df['HouseAge'] = df['YrSold'] - df['Year Built']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['Year Built']).astype(int)
    if neighborhood_median is not None:
        df['NeighborhoodPriceTier'] = df['Neighborhood'].map(neighborhood_median)
        df['NeighborhoodPriceTier'] = df['NeighborhoodPriceTier'].fillna(neighborhood_median.median())
    df['QualLivingArea'] = df['Gr Liv Area'] * df['Overall Qual']
    return df.fillna(0)

def add_fresh_features(df):
    df['LivAreaPerRoom'] = df['Gr Liv Area'] / df['TotRms AbvGrd'].replace(0, 1)
    df['AgeSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['TotalPorchSF'] = df['Open Porch SF'] + df['Enclosed Porch'] + df['3Ssn Porch'] + df['Screen Porch']
    df['GarageUtilScore'] = df['Garage Cars'] * df['Garage Area']
    return df.fillna(0)

neighborhood_median = train_df.groupby('Neighborhood')['SalePrice'].median()
train_df = add_features(train_df, neighborhood_median)
train_df = add_fresh_features(train_df)
test_df = add_features(test_df, neighborhood_median)
test_df = add_fresh_features(test_df)

sns.histplot(train_df['SalePrice'], bins=40, kde=True)
plt.title("Distribution of House Sale Prices")
plt.xlabel("Sale Price")
plt.ylabel("Count")
plt.show()

sns.countplot(x='Overall Qual', data=train_df)
plt.title("Count of Houses by Overall Quality")
plt.xlabel("Overall Quality Rating")
plt.ylabel("Number of Houses")
plt.show()

sns.scatterplot(x='Gr Liv Area', y='SalePrice', data=train_df)
plt.title("Sale Price vs. Ground Living Area")
plt.xlabel("Ground Living Area (sq ft)")
plt.ylabel("Sale Price")
plt.show()

top_neigh = train_df['Neighborhood'].value_counts().nlargest(10).index
sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df[train_df['Neighborhood'].isin(top_neigh)])
plt.title("Sale Price by Neighborhood (Top 10)")
plt.xticks(rotation=45)
plt.ylabel("Sale Price")
plt.show()

pairplot_features = ['SalePrice', 'Gr Liv Area', 'Garage Area', 'HouseAge', 'TotalBath', 'QualLivingArea']
sns.pairplot(train_df[pairplot_features], diag_kind='kde', corner=True)
plt.suptitle("Pairplot of Key Features", y=1.02)
plt.show()

correlations = train_df.corr(numeric_only=True)['SalePrice'].drop('SalePrice').sort_values(ascending=False)
print("\nTop Correlated Features with SalePrice:\n")
print(correlations.head(10))
print("\nLeast Correlated Features with SalePrice:\n")
print(correlations.tail(10))

features = [
    'Gr Liv Area', 'Garage Area', 'Year Built',
    'HouseAge', 'TotalBath', 'IsRemodeled',
    'NeighborhoodPriceTier', 'QualLivingArea',
    'LivAreaPerRoom', 'AgeSinceRemodel', 'TotalPorchSF', 'GarageUtilScore'
]

X_train = train_df[features]
y_train = train_df['SalePrice']
X_test = test_df[features]

def log_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, np.maximum(0, y_pred)))

log_rmse_scorer = make_scorer(log_rmse, greater_is_better=False)

gb = GradientBoostingRegressor(random_state=42)
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
}

grid_gb = GridSearchCV(gb, param_grid, scoring=log_rmse_scorer, cv=5)
grid_gb.fit(X_train, y_train)

print("\nRefit complete with new features.")
print("Best GB params:", grid_gb.best_params_)
print("Best GB Log-RMSE:", -grid_gb.best_score_)

preds = grid_gb.predict(X_test)
submission = pd.DataFrame({"Id": test_df["PID"], "SalePrice": preds})
submission.to_csv("submission.csv", index=False)
print("\nUpdated submission file created: submission.csv")

importances = grid_gb.best_estimator_.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop Feature Importances:")
print(feature_importance_df.head(10))

sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
plt.title("Top 10 Feature Importances")
plt.tight_layout()
plt.show()

NameError: name 'test_df' is not defined